# eICU APACHE III Variables Dataset Creation

This notebook creates APACHE III variables and calculates APACHE III scores for eICU sepsis cohort.

**Purpose:** Create a dataset matching the MIMIC-IV APACHE III dataset structure for external validation

**Dataset:** `my-new-project-473015.my_eicu_derived`

**Source Table:** `my_eicu_derived.sepsis3_cohort` (from 05_eicu_sepsis3_cohort_selection.ipynb)

---

## 📋 MIMIC-IV vs eICU Mapping

| MIMIC-IV Table | eICU Table | Notes |
|----------------|------------|-------|
| `stay_id` | `patientunitstayid` | ICU stay identifier |
| `subject_id` | `uniquepid` | Patient identifier |
| `icu_intime` (DATETIME) | offset = 0 | Time reference |
| `ventilation` | `respiratoryCare`, `apacheApsVar` | Ventilation status |
| `urine_output` | `physionet-data.eicu_crd_derived.pivoted_uo` | Urine output |
| `admissions.admission_type` | `patient.unitadmitsource` | Admission type |
| `labevents` | `lab` | Lab values |
| `vitalsign` | `vitalPeriodic`, `vitalAperiodic` | Vital signs |
| `bg` | `pivoted_bg` | Blood gas |
| `gcs` | `physionet-data.eicu_crd_derived.pivoted_gcs` | GCS scores |

---

## 🔧 Key Differences in Time Handling

- **MIMIC-IV:** Uses DATETIME for timestamps
- **eICU:** Uses offset in **minutes** from ICU admission (offset = 0)
  - Positive offset = after ICU admission
  - Negative offset = before ICU admission
  - 24 hours = 1440 minutes

---

## Analysis Steps

**STEP 1:** Setup  
**STEP 2:** Add Ventilation data  
**STEP 3:** Add Urine Output data  
**STEP 4:** Add Admission Type  
**STEP 5:** Add Acute Renal Failure (ARF)  
**STEP 6:** Extract Vital Signs (Long Format)  
**STEP 7:** Extract FiO2 (Long Format)  
**STEP 8:** Extract Lab Values (Long Format)  
**STEP 9:** Extract Blood Gas Values (Long Format)  
**STEP 10:** Extract GCS Components (Long Format)  
**STEP 11:** Combine All Tables  
**STEP 12:** Add Age and Comorbidity  
**STEP 13:** Aggregate Values (Wide Format)  
**STEP 14:** Calculate APACHE III Scores  
**STEP 15:** Combine with Vital Signs Data  


## STEP 1: Setup

In [63]:
# Import libraries
from google.cloud import bigquery
import pandas as pd
from datetime import datetime

# Initialize BigQuery client
client = bigquery.Client(project='my-new-project-473015')

# Configuration
PROJECT_ID = 'my-new-project-473015'
DATASET_ID = 'my_eicu_derived'
SOURCE_DATASET = 'physionet-data.eicu_crd'
DERIVED_DATASET = 'physionet-data.eicu_crd_derived'

# Time constants (in minutes for eICU)
HOURS_24 = 1440  # 24 * 60 minutes

print("="*70)
print("eICU APACHE III Variables Dataset Creation")
print(f"Target dataset: {PROJECT_ID}.{DATASET_ID}")
print(f"Source dataset: {SOURCE_DATASET}")
print(f"Derived dataset: {DERIVED_DATASET}")
print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*70)
print("\n✅ Setup complete! Ready to create APACHE III dataset.")

eICU APACHE III Variables Dataset Creation
Target dataset: my-new-project-473015.my_eicu_derived
Source dataset: physionet-data.eicu_crd
Derived dataset: physionet-data.eicu_crd_derived
Timestamp: 2026-01-29 19:08:02

✅ Setup complete! Ready to create APACHE III dataset.


## STEP 2: Check sepsis3_cohort Table Structure

In [64]:
%%bigquery
SELECT
  column_name,
  data_type,
  is_nullable
FROM `my-new-project-473015.my_eicu_derived.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name = 'sepsis3_cohort'
ORDER BY ordinal_position

,column_name,data_type,is_nullable
0,patientunitstayid,INT64,YES
1,uniquepid,STRING,YES
2,hospitalid,INT64,YES
3,age,INT64,YES
4,gender,STRING,YES
5,ethnicity,STRING,YES
6,unittype,STRING,YES
7,unitvisitnumber,INT64,YES
8,hospitaladmitoffset,INT64,YES
9,hospitaldischargeoffset,INT64,YES


## STEP 3: Adding Ventilation Data

Goal: Add ventilation data to sepsis3_cohort
- Determine if invasive ventilation was used within 24 hours of ICU admission
- eICU time reference: offset = 0 is ICU admission, 1440 = 24 hours after

**eICU Sources:**
- `respiratoryCare` table: contains ventilation settings
- `apacheApsVar` table: contains `intubated` flag
- `physionet-data.eicu_crd_derived.ventilation_events`: derived ventilation events

In [65]:
%%bigquery
CREATE OR REPLACE TABLE `my-new-project-473015.my_eicu_derived.sepsis3_cohort_with_vent` AS
WITH vent_24h AS (
  SELECT
    s.patientunitstayid,
    -- Check if any invasive ventilation occurred within 24 hours
    CASE
      WHEN MAX(CASE WHEN a.intubated = 1 THEN 1 ELSE 0 END) = 1 THEN TRUE
      WHEN MAX(CASE
        WHEN rc.airwaytype IN ('Oral ETT', 'Nasal ETT', 'Tracheostomy')
        AND rc.ventstartoffset >= 0
        AND rc.ventstartoffset < 1440 THEN 1
        ELSE 0 END) = 1 THEN TRUE
      ELSE FALSE
    END AS vent,
    -- Get earliest ventilation start time within 24h window (in minutes)
    MIN(CASE
      WHEN rc.ventstartoffset >= 0 AND rc.ventstartoffset < 1440
      THEN rc.ventstartoffset
      ELSE NULL END) AS vent_startoffset,
    -- Get latest ventilation end time within 24h window
    MAX(CASE
      WHEN rc.ventendoffset >= 0 AND rc.ventendoffset < 1440
      THEN rc.ventendoffset
      ELSE NULL END) AS vent_endoffset
  FROM `my-new-project-473015.my_eicu_derived.sepsis3_cohort` s
  LEFT JOIN `physionet-data.eicu_crd.apacheapsvar` a
    ON s.patientunitstayid = a.patientunitstayid
  LEFT JOIN `physionet-data.eicu_crd.respiratorycare` rc
    ON s.patientunitstayid = rc.patientunitstayid
  GROUP BY s.patientunitstayid
)
SELECT
  s.*,
  COALESCE(v.vent, FALSE) AS vent,
  v.vent_startoffset,
  v.vent_endoffset
FROM `my-new-project-473015.my_eicu_derived.sepsis3_cohort` s
LEFT JOIN vent_24h v
  ON s.patientunitstayid = v.patientunitstayid;

SELECT 'Table sepsis3_cohort_with_vent created successfully!' AS status;

,status
0,Table sepsis3_cohort_with_vent created success...


In [66]:
%%bigquery
-- Verify ventilation data
SELECT
  COUNT(*) AS total_patients,
  SUM(CASE WHEN vent = TRUE THEN 1 ELSE 0 END) AS ventilated_count,
  ROUND(100.0 * SUM(CASE WHEN vent = TRUE THEN 1 ELSE 0 END) / COUNT(*), 1) AS ventilated_pct
FROM `my-new-project-473015.my_eicu_derived.sepsis3_cohort_with_vent`

,total_patients,ventilated_count,ventilated_pct
0,31410,9812,31.2


## STEP 4: Adding Urine Output Data

Goal: Add urine output data to cohort
- Calculate total urine output within 24 hours of ICU admission
- Variable name: uop (urine output in mL)

**eICU Source:** `physionet-data.eicu_crd_derived.pivoted_uo`

In [67]:
%%bigquery
CREATE OR REPLACE TABLE `my-new-project-473015.my_eicu_derived.sepsis3_cohort_with_vent_uop` AS
WITH uop_24h AS (
  SELECT
    s.patientunitstayid,
    -- Sum all urine output within 24 hours of ICU admission
    -- eICU: chartoffset >= 0 AND chartoffset < 1440 (24 hours in minutes)
    SUM(u.urineoutput) AS uop
  FROM `my-new-project-473015.my_eicu_derived.sepsis3_cohort_with_vent` s
  LEFT JOIN `physionet-data.eicu_crd_derived.pivoted_uo` u
    ON s.patientunitstayid = u.patientunitstayid
    AND u.chartoffset >= 0
    AND u.chartoffset < 1440  -- 24 hours in minutes
  GROUP BY s.patientunitstayid
)
SELECT
  s.*,
  COALESCE(u.uop, 0) AS uop
FROM `my-new-project-473015.my_eicu_derived.sepsis3_cohort_with_vent` s
LEFT JOIN uop_24h u
  ON s.patientunitstayid = u.patientunitstayid;

SELECT 'Table sepsis3_cohort_with_vent_uop created successfully!' AS status;

,status
0,Table sepsis3_cohort_with_vent_uop created suc...


In [68]:
%%bigquery
-- Verify urine output data
SELECT
  COUNT(*) AS total_patients,
  SUM(CASE WHEN uop > 0 THEN 1 ELSE 0 END) AS has_uop,
  ROUND(AVG(uop), 1) AS avg_uop_ml,
  ROUND(MIN(uop), 1) AS min_uop,
  ROUND(MAX(uop), 1) AS max_uop
FROM `my-new-project-473015.my_eicu_derived.sepsis3_cohort_with_vent_uop`

,total_patients,has_uop,avg_uop_ml,min_uop,max_uop
0,31410,22980,1230.4,0.0,89175.0


## STEP 5: Add Admission Type

Goal: Add admission type to the cohort
- Classify as 'elective' or 'emergency' for APACHE III scoring
- Variable name: admit

**eICU:** Uses `unitadmitsource` and `apacheadmissiondx` from patient table

In [69]:
%%bigquery
SELECT
  unitadmitsource,
  COUNT(*) AS count
FROM `physionet-data.eicu_crd.patient`
GROUP BY unitadmitsource
ORDER BY count DESC

,unitadmitsource,count
0,Emergency Department,89594
1,Floor,24368
2,Operating Room,24305
3,ICU to SDU,13827
4,Direct Admit,12672
5,Recovery Room,7844
6,Acute Care/Floor,5604
7,Step-Down Unit (SDU),5450
8,ICU,5439
9,Other Hospital,4323


In [70]:
%%bigquery
CREATE OR REPLACE TABLE `my-new-project-473015.my_eicu_derived.sepsis3_cohort_with_vent_uop_admit` AS
WITH admit_type AS (
  SELECT
    s.patientunitstayid,
    -- Classify admission type for APACHE III
    -- Priority 1: Use electivesurgery from apachepredvar if available
    -- Priority 2: Fallback to unitadmitsource logic
    CASE
      -- First, use apachepredvar.electivesurgery if available
      WHEN a.electivesurgery = 1 THEN 'elective'
      WHEN a.electivesurgery = 0 THEN 'emergency'
      -- Fallback: use unitadmitsource if electivesurgery is NULL
      WHEN p.unitadmitsource IN ('Operating Room', 'Recovery Room', 'PACU') THEN 'elective'
      ELSE 'emergency'
    END AS admit
  FROM `my-new-project-473015.my_eicu_derived.sepsis3_cohort_with_vent_uop` s
  LEFT JOIN `physionet-data.eicu_crd.apachepredvar` a
    ON s.patientunitstayid = a.patientunitstayid
  LEFT JOIN `physionet-data.eicu_crd.patient` p
    ON s.patientunitstayid = p.patientunitstayid
)
SELECT
  s.*,
  COALESCE(a.admit, 'emergency') AS admit
FROM `my-new-project-473015.my_eicu_derived.sepsis3_cohort_with_vent_uop` s
LEFT JOIN admit_type a
  ON s.patientunitstayid = a.patientunitstayid;

SELECT 'Table sepsis3_cohort_with_vent_uop_admit created successfully!' AS status;

,status
0,Table sepsis3_cohort_with_vent_uop_admit creat...


In [71]:
%%bigquery
-- Verify admission type distribution
SELECT
  admit,
  COUNT(*) AS count,
  ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 1) AS percentage
FROM `my-new-project-473015.my_eicu_derived.sepsis3_cohort_with_vent_uop_admit`
GROUP BY admit

,admit,count,percentage
0,emergency,24885,79.2
1,elective,6525,20.8


## STEP 6: Adding Acute Renal Failure (ARF)

Goal: Add acute renal failure (ARF) flag to the cohort

ARF Definition for APACHE III:
- Serum Creatinine (SCr) >= 1.5 mg/dL
- AND Urine Output < 410 mL (24 hours)

**eICU Source:** `lab` table for creatinine values

In [72]:
%%bigquery
CREATE OR REPLACE TABLE `my-new-project-473015.my_eicu_derived.sepsis3_cohort_with_vent_uop_admit_arf` AS
WITH scr_24h AS (
  SELECT
    s.patientunitstayid,
    -- Get maximum serum creatinine within 24 hours
    MAX(CAST(l.labresult AS FLOAT64)) AS scr_max
  FROM `my-new-project-473015.my_eicu_derived.sepsis3_cohort_with_vent_uop_admit` s
  LEFT JOIN `physionet-data.eicu_crd.lab` l
    ON s.patientunitstayid = l.patientunitstayid
  WHERE
    -- Filter for creatinine lab tests
    LOWER(l.labname) LIKE '%creatinine%'
    AND LOWER(l.labname) NOT LIKE '%clear%'  -- exclude creatinine clearance
    -- Within 24 hours of ICU admission
    AND l.labresultoffset >= 0
    AND l.labresultoffset < 1440  -- 24 hours in minutes
    -- Valid values only
    AND l.labresult IS NOT NULL
    AND SAFE_CAST(l.labresult AS FLOAT64) > 0
    AND SAFE_CAST(l.labresult AS FLOAT64) < 20  -- exclude extreme outliers
  GROUP BY s.patientunitstayid
),
arf_calculation AS (
  SELECT
    s.patientunitstayid,
    scr.scr_max,
    -- ARF = TRUE if SCr >= 1.5 AND UOP < 410
    CASE
      WHEN scr.scr_max >= 1.5 AND s.uop < 410 THEN TRUE
      ELSE FALSE
    END AS arf
  FROM `my-new-project-473015.my_eicu_derived.sepsis3_cohort_with_vent_uop_admit` s
  LEFT JOIN scr_24h scr
    ON s.patientunitstayid = scr.patientunitstayid
)
SELECT
  s.*,
  a.scr_max,
  COALESCE(a.arf, FALSE) AS arf
FROM `my-new-project-473015.my_eicu_derived.sepsis3_cohort_with_vent_uop_admit` s
LEFT JOIN arf_calculation a
  ON s.patientunitstayid = a.patientunitstayid;

SELECT 'Table sepsis3_cohort_with_vent_uop_admit_arf created successfully!' AS status;

,status
0,Table sepsis3_cohort_with_vent_uop_admit_arf c...


In [73]:
%%bigquery
-- Verify ARF data
SELECT
  COUNT(*) AS total_patients,
  SUM(CASE WHEN arf = TRUE THEN 1 ELSE 0 END) AS arf_count,
  ROUND(100.0 * SUM(CASE WHEN arf = TRUE THEN 1 ELSE 0 END) / COUNT(*), 1) AS arf_pct,
  ROUND(AVG(scr_max), 2) AS avg_scr
FROM `my-new-project-473015.my_eicu_derived.sepsis3_cohort_with_vent_uop_admit_arf`

,total_patients,arf_count,arf_pct,avg_scr
0,31410,5241,16.7,1.79


## STEP 7: Extract Vital Signs (Long Format)

Goal: Extract all vital sign measurements within 24 hours of ICU admission
- Create long-format table with all measurements over time
- Variables: hr, map, temp, rr

**eICU Source:** `my_eicu_derived.vitalsign` (created in 04_eicu_sepsis3_table_creation.ipynb)

In [74]:
%%bigquery
CREATE OR REPLACE TABLE `my-new-project-473015.my_eicu_derived.sepsis3_vitals_24h_raw` AS
WITH cohort AS (
  SELECT
    patientunitstayid
  FROM `my-new-project-473015.my_eicu_derived.sepsis3_cohort_with_vent_uop_admit_arf`
)
-- Heart Rate
SELECT
  c.patientunitstayid,
  v.chartoffset AS observationoffset,
  'hr' AS variable_name,
  v.heartrate AS value
FROM cohort c
INNER JOIN `physionet-data.eicu_crd_derived.pivoted_vital` v
  ON c.patientunitstayid = v.patientunitstayid
  AND v.chartoffset >= 0
  AND v.chartoffset < 1440
WHERE v.heartrate IS NOT NULL
  AND v.heartrate > 0
  AND v.heartrate <= 300

UNION ALL

-- Mean Arterial Pressure (prefer invasive, fallback to non-invasive)
SELECT
  c.patientunitstayid,
  v.chartoffset AS observationoffset,
  'map' AS variable_name,
  COALESCE(v.ibp_mean, v.nibp_mean) AS value
FROM cohort c
INNER JOIN `physionet-data.eicu_crd_derived.pivoted_vital` v
  ON c.patientunitstayid = v.patientunitstayid
  AND v.chartoffset >= 0
  AND v.chartoffset < 1440
WHERE COALESCE(v.ibp_mean, v.nibp_mean) IS NOT NULL
  AND COALESCE(v.ibp_mean, v.nibp_mean) > 0
  AND COALESCE(v.ibp_mean, v.nibp_mean) <= 250

UNION ALL

-- Temperature (Celsius)
SELECT
  c.patientunitstayid,
  v.chartoffset AS observationoffset,
  'temp' AS variable_name,
  -- Convert Fahrenheit to Celsius if needed
  CASE
    WHEN v.temperature > 50 THEN (v.temperature - 32) * 5 / 9  -- Fahrenheit to Celsius
    ELSE v.temperature  -- Already Celsius
  END AS value
FROM cohort c
INNER JOIN `physionet-data.eicu_crd_derived.pivoted_vital` v
  ON c.patientunitstayid = v.patientunitstayid
  AND v.chartoffset >= 0
  AND v.chartoffset < 1440
WHERE v.temperature IS NOT NULL
  AND (
    (v.temperature > 25 AND v.temperature < 45)  -- Valid Celsius range
    OR (v.temperature > 77 AND v.temperature < 113)  -- Valid Fahrenheit range (converts to 25-45°C)
  )

UNION ALL

-- Respiratory Rate
SELECT
  c.patientunitstayid,
  v.chartoffset AS observationoffset,
  'rr' AS variable_name,
  v.respiratoryrate AS value
FROM cohort c
INNER JOIN `physionet-data.eicu_crd_derived.pivoted_vital` v
  ON c.patientunitstayid = v.patientunitstayid
  AND v.chartoffset >= 0
  AND v.chartoffset < 1440
WHERE v.respiratoryrate IS NOT NULL
  AND v.respiratoryrate > 0
  AND v.respiratoryrate <= 70

ORDER BY patientunitstayid, observationoffset, variable_name;

SELECT 'Table sepsis3_vitals_24h_raw created successfully!' AS status;

,status
0,Table sepsis3_vitals_24h_raw created successfu...


In [75]:
%%bigquery
-- Verify vital signs data
SELECT
  variable_name,
  COUNT(*) AS count,
  COUNT(DISTINCT patientunitstayid) AS patients,
  ROUND(AVG(value), 2) AS avg_value,
  ROUND(MIN(value), 2) AS min_value,
  ROUND(MAX(value), 2) AS max_value
FROM `my-new-project-473015.my_eicu_derived.sepsis3_vitals_24h_raw`
GROUP BY variable_name

,variable_name,count,patients,avg_value,min_value,max_value
0,hr,1284904,30768,88.35,25.0,225.0
1,rr,1163469,30013,20.16,1.0,60.0
2,map,1187027,30826,77.47,1.0,250.0
3,temp,346773,30972,36.87,25.4,42.1


## STEP 8: Extract FiO2 (Long Format)

Goal: Extract all FiO2 measurements within 24 hours of ICU admission
- Variable: fio2 (fraction of inspired oxygen, as percentage 0-100)

**eICU Source:** `respiratorycharting` and `apacheapsvar`

In [76]:
%%bigquery
CREATE OR REPLACE TABLE `my-new-project-473015.my_eicu_derived.sepsis3_fio2_24h_raw` AS
WITH cohort AS (
  SELECT
    patientunitstayid
  FROM `my-new-project-473015.my_eicu_derived.sepsis3_cohort_with_vent_uop_admit_arf`
),
fio2_raw AS (
  -- Extract FiO2 from respiratoryCharting
  SELECT
    c.patientunitstayid,
    rc.respchartoffset AS observationoffset,
    'fio2' AS variable_name,
    'respiratorycharting' AS source,
    CASE
      WHEN SAFE_CAST(rc.respchartvalue AS FLOAT64) <= 1
        THEN SAFE_CAST(rc.respchartvalue AS FLOAT64) * 100
      ELSE SAFE_CAST(rc.respchartvalue AS FLOAT64)
    END AS value
  FROM cohort c
  INNER JOIN `physionet-data.eicu_crd.respiratorycharting` rc
    ON c.patientunitstayid = rc.patientunitstayid
    AND rc.respchartoffset >= 0
    AND rc.respchartoffset < 1440
  WHERE LOWER(rc.respchartvaluelabel) LIKE '%fio2%'
    AND rc.respchartvalue IS NOT NULL
    AND SAFE_CAST(rc.respchartvalue AS FLOAT64) > 0

  UNION ALL

  -- Extract FiO2 from apacheApsVar (single value per patient)
  SELECT
    c.patientunitstayid,
    0 AS observationoffset,
    'fio2' AS variable_name,
    'apacheapsvar' AS source,
    CASE
      WHEN a.fio2 <= 1 THEN a.fio2 * 100
      ELSE a.fio2
    END AS value
  FROM cohort c
  INNER JOIN `physionet-data.eicu_crd.apacheapsvar` a
    ON c.patientunitstayid = a.patientunitstayid
  WHERE a.fio2 IS NOT NULL
    AND a.fio2 > 0
)
-- Filter for valid FiO2 range (21-100%)
SELECT *
FROM fio2_raw
WHERE value >= 21 AND value <= 100
ORDER BY patientunitstayid, observationoffset;

SELECT 'Table sepsis3_fio2_24h_raw created successfully!' AS status;

,status
0,Table sepsis3_fio2_24h_raw created successfully!


In [77]:
%%bigquery
-- Verify FiO2 data
SELECT
  source,
  COUNT(*) AS count,
  COUNT(DISTINCT patientunitstayid) AS patients,
  ROUND(AVG(value), 2) AS avg_fio2,
  ROUND(MIN(value), 2) AS min_fio2,
  ROUND(MAX(value), 2) AS max_fio2
FROM `my-new-project-473015.my_eicu_derived.sepsis3_fio2_24h_raw`
GROUP BY source

,source,count,patients,avg_fio2,min_fio2,max_fio2
0,apacheapsvar,12770,12770,61.28,21.0,100.0
1,respiratorycharting,251819,18962,52.79,21.0,100.0


## STEP 9: Extract Lab Values (Long Format)

Goal: Extract all lab value measurements within ±24 hours of ICU admission
- Variables: sodium, bun, hct, wbc, albumin, bili, glucose, scr

**eICU Source:** `physionet-data.eicu_crd.lab`

**Note:** Time window is -24h to +24h (offset -1440 to +1440) to match MIMIC-IV approach

In [78]:
%%bigquery
CREATE OR REPLACE TABLE `my-new-project-473015.my_eicu_derived.sepsis3_labs_24h_raw` AS
WITH cohort AS (
  SELECT
    patientunitstayid
  FROM `my-new-project-473015.my_eicu_derived.sepsis3_cohort_with_vent_uop_admit_arf`
)
-- Bilirubin (total)
SELECT
  c.patientunitstayid,
  l.labresultoffset AS observationoffset,
  'bili' AS variable_name,
  SAFE_CAST(l.labresult AS FLOAT64) AS value
FROM cohort c
INNER JOIN `physionet-data.eicu_crd.lab` l
  ON c.patientunitstayid = l.patientunitstayid
  AND l.labresultoffset >= -1440  -- 24h before
  AND l.labresultoffset < 1440     -- 24h after
WHERE LOWER(l.labname) = 'total bilirubin'
  AND l.labresult IS NOT NULL
  AND SAFE_CAST(l.labresult AS FLOAT64) >= 0
  AND SAFE_CAST(l.labresult AS FLOAT64) <= 50

UNION ALL

-- BUN
SELECT
  c.patientunitstayid,
  l.labresultoffset AS observationoffset,
  'bun' AS variable_name,
  SAFE_CAST(l.labresult AS FLOAT64) AS value
FROM cohort c
INNER JOIN `physionet-data.eicu_crd.lab` l
  ON c.patientunitstayid = l.patientunitstayid
  AND l.labresultoffset >= -1440
  AND l.labresultoffset < 1440
WHERE LOWER(l.labname) = 'bun'
  AND l.labresult IS NOT NULL
  AND SAFE_CAST(l.labresult AS FLOAT64) >= 0
  AND SAFE_CAST(l.labresult AS FLOAT64) <= 300

UNION ALL

-- Sodium
SELECT
  c.patientunitstayid,
  l.labresultoffset AS observationoffset,
  'sodium' AS variable_name,
  SAFE_CAST(l.labresult AS FLOAT64) AS value
FROM cohort c
INNER JOIN `physionet-data.eicu_crd.lab` l
  ON c.patientunitstayid = l.patientunitstayid
  AND l.labresultoffset >= -1440
  AND l.labresultoffset < 1440
WHERE LOWER(l.labname) = 'sodium'
  AND l.labresult IS NOT NULL
  AND SAFE_CAST(l.labresult AS FLOAT64) >= 100
  AND SAFE_CAST(l.labresult AS FLOAT64) <= 200

UNION ALL

-- Glucose
SELECT
  c.patientunitstayid,
  l.labresultoffset AS observationoffset,
  'glucose' AS variable_name,
  SAFE_CAST(l.labresult AS FLOAT64) AS value
FROM cohort c
INNER JOIN `physionet-data.eicu_crd.lab` l
  ON c.patientunitstayid = l.patientunitstayid
  AND l.labresultoffset >= -1440
  AND l.labresultoffset < 1440
WHERE LOWER(l.labname) = 'glucose'
  AND l.labresult IS NOT NULL
  AND SAFE_CAST(l.labresult AS FLOAT64) >= 0
  AND SAFE_CAST(l.labresult AS FLOAT64) <= 1000

UNION ALL

-- Creatinine (updated: upper limit 30)
SELECT
  c.patientunitstayid,
  l.labresultoffset AS observationoffset,
  'scr' AS variable_name,
  SAFE_CAST(l.labresult AS FLOAT64) AS value
FROM cohort c
INNER JOIN `physionet-data.eicu_crd.lab` l
  ON c.patientunitstayid = l.patientunitstayid
  AND l.labresultoffset >= -1440
  AND l.labresultoffset < 1440
WHERE LOWER(l.labname) LIKE '%creatinine%'
  AND LOWER(l.labname) NOT LIKE '%clearance%'
  AND l.labresult IS NOT NULL
  AND SAFE_CAST(l.labresult AS FLOAT64) >= 0
  AND SAFE_CAST(l.labresult AS FLOAT64) <= 30

UNION ALL

-- Hematocrit (updated: lower limit 10, upper limit 75)
SELECT
  c.patientunitstayid,
  l.labresultoffset AS observationoffset,
  'hct' AS variable_name,
  SAFE_CAST(l.labresult AS FLOAT64) AS value
FROM cohort c
INNER JOIN `physionet-data.eicu_crd.lab` l
  ON c.patientunitstayid = l.patientunitstayid
  AND l.labresultoffset >= -1440
  AND l.labresultoffset < 1440
WHERE LOWER(l.labname) = 'hct'
  AND l.labresult IS NOT NULL
  AND SAFE_CAST(l.labresult AS FLOAT64) >= 10  --
  AND SAFE_CAST(l.labresult AS FLOAT64) <= 75  --

UNION ALL

-- WBC
SELECT
  c.patientunitstayid,
  l.labresultoffset AS observationoffset,
  'wbc' AS variable_name,
  SAFE_CAST(l.labresult AS FLOAT64) AS value
FROM cohort c
INNER JOIN `physionet-data.eicu_crd.lab` l
  ON c.patientunitstayid = l.patientunitstayid
  AND l.labresultoffset >= -1440
  AND l.labresultoffset < 1440
WHERE LOWER(l.labname) IN ('wbc x 1000', 'wbc')
  AND l.labresult IS NOT NULL
  AND SAFE_CAST(l.labresult AS FLOAT64) >= 0
  AND SAFE_CAST(l.labresult AS FLOAT64) <= 500  --

UNION ALL

-- Albumin
SELECT
  c.patientunitstayid,
  l.labresultoffset AS observationoffset,
  'albumin' AS variable_name,
  SAFE_CAST(l.labresult AS FLOAT64) AS value
FROM cohort c
INNER JOIN `physionet-data.eicu_crd.lab` l
  ON c.patientunitstayid = l.patientunitstayid
  AND l.labresultoffset >= -1440
  AND l.labresultoffset < 1440
WHERE LOWER(l.labname) = 'albumin'
  AND l.labresult IS NOT NULL
  AND SAFE_CAST(l.labresult AS FLOAT64) >= 0
  AND SAFE_CAST(l.labresult AS FLOAT64) <= 10

ORDER BY patientunitstayid, observationoffset, variable_name;

SELECT 'Labs table created successfully (time window: -24h to +24h)!' AS status;

,status
0,Labs table created successfully (time window: ...


In [79]:
%%bigquery
-- Verify lab data
SELECT
  variable_name,
  COUNT(*) AS count,
  COUNT(DISTINCT patientunitstayid) AS patients,
  ROUND(AVG(value), 2) AS avg_value,
  ROUND(MIN(value), 2) AS min_value,
  ROUND(MAX(value), 2) AS max_value
FROM `my-new-project-473015.my_eicu_derived.sepsis3_labs_24h_raw`
GROUP BY variable_name
ORDER BY variable_name

,variable_name,count,patients,avg_value,min_value,max_value
0,albumin,37454,22531,2.88,0.20,6.2
1,bili,35239,21870,1.43,0.00,48.0
2,bun,77764,30709,31.59,0.00,298.0
3,glucose,90345,30684,156.31,2.00,998.0
4,hct,93501,30956,32.05,10.00,72.4
5,scr,78707,30715,1.86,0.09,30.0
6,sodium,101386,30771,138.39,100.00,195.0
7,wbc,69415,30870,13.51,0.00,498.8


## STEP 10: Extract Blood Gas Values (Long Format)

Goal: Extract all blood gas measurements within ±24 hours of ICU admission
- Variables: ph, pao2, pco2, aa_grad

**eICU Source:** `physionet-data.eicu_crd_derived.pivoted_bg`

In [80]:
%%bigquery
SELECT column_name, data_type
FROM `physionet-data.eicu_crd_derived.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name = 'pivoted_bg'
ORDER BY ordinal_position

,column_name,data_type
0,patientunitstayid,INT64
1,chartoffset,INT64
2,fio2,FLOAT64
3,pao2,FLOAT64
4,paco2,FLOAT64
5,pH,FLOAT64
6,aniongap,FLOAT64
7,basedeficit,FLOAT64
8,baseexcess,FLOAT64
9,peep,FLOAT64


In [81]:
%%bigquery
CREATE OR REPLACE TABLE `my-new-project-473015.my_eicu_derived.sepsis3_bloodgas_24h_raw` AS
WITH cohort AS (
  SELECT
    patientunitstayid
  FROM `my-new-project-473015.my_eicu_derived.sepsis3_cohort_with_vent_uop_admit_arf`
),
-- Get ABG reference values from apacheApsVar
apache_abg AS (
  SELECT
    patientunitstayid,
    pao2 AS apache_pao2,
    pco2 AS apache_pco2,
    ph AS apache_ph
  FROM `physionet-data.eicu_crd.apacheapsvar`
)
-- pH
SELECT
  c.patientunitstayid,
  bg.chartoffset AS observationoffset,
  'ph' AS variable_name,
  bg.ph AS value
FROM cohort c
INNER JOIN `physionet-data.eicu_crd_derived.pivoted_bg` bg
  ON c.patientunitstayid = bg.patientunitstayid
  AND bg.chartoffset >= -1440
  AND bg.chartoffset < 1440
LEFT JOIN apache_abg a
  ON c.patientunitstayid = a.patientunitstayid
WHERE bg.ph IS NOT NULL
  AND bg.ph >= 6.5
  AND bg.ph <= 8.0

UNION ALL

-- PaO2 (exclude if lower than apache ABG reference = likely VBG)
SELECT
  c.patientunitstayid,
  bg.chartoffset AS observationoffset,
  'pao2' AS variable_name,
  bg.pao2 AS value
FROM cohort c
INNER JOIN `physionet-data.eicu_crd_derived.pivoted_bg` bg
  ON c.patientunitstayid = bg.patientunitstayid
  AND bg.chartoffset >= -1440
  AND bg.chartoffset < 1440
LEFT JOIN apache_abg a
  ON c.patientunitstayid = a.patientunitstayid
WHERE bg.pao2 IS NOT NULL
  AND bg.pao2 > 0
  AND bg.pao2 <= 700
  -- Exclude VBG: if bg.pao2 < apache_pao2, likely VBG
  AND (a.apache_pao2 IS NULL OR bg.pao2 >= a.apache_pao2 * 0.9)

UNION ALL

-- PCO2 (exclude if higher than apache ABG reference = likely VBG)
SELECT
  c.patientunitstayid,
  bg.chartoffset AS observationoffset,
  'pco2' AS variable_name,
  bg.paco2 AS value
FROM cohort c
INNER JOIN `physionet-data.eicu_crd_derived.pivoted_bg` bg
  ON c.patientunitstayid = bg.patientunitstayid
  AND bg.chartoffset >= -1440
  AND bg.chartoffset < 1440
LEFT JOIN apache_abg a
  ON c.patientunitstayid = a.patientunitstayid
WHERE bg.paco2 IS NOT NULL
  AND bg.paco2 > 0
  AND bg.paco2 <= 200
  -- Exclude VBG: if bg.paco2 > apache_pco2, likely VBG
  AND (a.apache_pco2 IS NULL OR bg.paco2 <= a.apache_pco2 * 1.1)

UNION ALL

-- A-a gradient (calculated, only from validated ABG values)
SELECT
  c.patientunitstayid,
  bg.chartoffset AS observationoffset,
  'aa_grad' AS variable_name,
  bg.fio2 * 713 - (bg.paco2 / 0.8) - bg.pao2 AS value
FROM cohort c
INNER JOIN `physionet-data.eicu_crd_derived.pivoted_bg` bg
  ON c.patientunitstayid = bg.patientunitstayid
  AND bg.chartoffset >= -1440
  AND bg.chartoffset < 1440
LEFT JOIN apache_abg a
  ON c.patientunitstayid = a.patientunitstayid
WHERE bg.fio2 IS NOT NULL
  AND bg.paco2 IS NOT NULL
  AND bg.pao2 IS NOT NULL
  AND bg.fio2 >= 0.21
  AND bg.fio2 <= 1.0
  AND bg.paco2 > 0
  AND bg.pao2 > 0
  -- Exclude VBG using both criteria
  AND (a.apache_pao2 IS NULL OR bg.pao2 >= a.apache_pao2 * 0.9)
  AND (a.apache_pco2 IS NULL OR bg.paco2 <= a.apache_pco2 * 1.1)
  -- Valid A-a gradient range
  AND (bg.fio2 * 713 - (bg.paco2 / 0.8) - bg.pao2) >= 0
  AND (bg.fio2 * 713 - (bg.paco2 / 0.8) - bg.pao2) <= 700

ORDER BY patientunitstayid, observationoffset, variable_name;

SELECT 'Table sepsis3_bloodgas_24h_raw created successfully (ABG filtered)!' AS status;

,status
0,Table sepsis3_bloodgas_24h_raw created success...


In [82]:
%%bigquery
-- Verify blood gas data
SELECT
  variable_name,
  COUNT(*) AS count,
  COUNT(DISTINCT patientunitstayid) AS patients,
  ROUND(AVG(value), 2) AS avg_value,
  ROUND(MIN(value), 2) AS min_value,
  ROUND(MAX(value), 2) AS max_value
FROM `my-new-project-473015.my_eicu_derived.sepsis3_bloodgas_24h_raw`
GROUP BY variable_name
ORDER BY variable_name

,variable_name,count,patients,avg_value,min_value,max_value
0,aa_grad,24482,12977,252.81,0.00,650.38
1,pao2,61140,20846,172.03,15.00,700.00
2,pco2,42941,13252,40.85,6.90,129.80
3,ph,73242,20150,7.34,6.63,8.00


## STEP 11: Extract GCS Components (Long Format)

Goal: Extract all GCS measurements within 24 hours of ICU admission
- Variables: gcs_eye, gcs_motor, gcs_verbal

**eICU Source:** `physionet-data.eicu_crd_derived.pivoted_gcs`

In [83]:
%%bigquery
CREATE OR REPLACE TABLE `my-new-project-473015.my_eicu_derived.sepsis3_gcs_24h_raw` AS
WITH cohort AS (
  SELECT
    patientunitstayid
  FROM `my-new-project-473015.my_eicu_derived.sepsis3_cohort_with_vent_uop_admit_arf`
)
-- GCS Eye
SELECT
  c.patientunitstayid,
  g.chartoffset AS observationoffset,
  'gcs_eye' AS variable_name,
  CAST(g.gcseyes AS FLOAT64) AS value
FROM cohort c
INNER JOIN `physionet-data.eicu_crd_derived.pivoted_gcs` g
  ON c.patientunitstayid = g.patientunitstayid
  AND g.chartoffset >= 0
  AND g.chartoffset < 1440
WHERE g.gcseyes IS NOT NULL
  AND g.gcseyes >= 1
  AND g.gcseyes <= 4

UNION ALL

-- GCS Motor
SELECT
  c.patientunitstayid,
  g.chartoffset AS observationoffset,
  'gcs_motor' AS variable_name,
  CAST(g.gcsmotor AS FLOAT64) AS value
FROM cohort c
INNER JOIN `physionet-data.eicu_crd_derived.pivoted_gcs` g
  ON c.patientunitstayid = g.patientunitstayid
  AND g.chartoffset >= 0
  AND g.chartoffset < 1440
WHERE g.gcsmotor IS NOT NULL
  AND g.gcsmotor >= 1
  AND g.gcsmotor <= 6

UNION ALL

-- GCS Verbal
SELECT
  c.patientunitstayid,
  g.chartoffset AS observationoffset,
  'gcs_verbal' AS variable_name,
  CAST(g.gcsverbal AS FLOAT64) AS value
FROM cohort c
INNER JOIN `physionet-data.eicu_crd_derived.pivoted_gcs` g
  ON c.patientunitstayid = g.patientunitstayid
  AND g.chartoffset >= 0
  AND g.chartoffset < 1440
WHERE g.gcsverbal IS NOT NULL
  AND g.gcsverbal >= 1
  AND g.gcsverbal <= 5

ORDER BY patientunitstayid, observationoffset, variable_name;

SELECT 'Table sepsis3_gcs_24h_raw created successfully!' AS status;

,status
0,Table sepsis3_gcs_24h_raw created successfully!


In [84]:
%%bigquery
-- Verify GCS data
SELECT
  variable_name,
  COUNT(*) AS count,
  COUNT(DISTINCT patientunitstayid) AS patients,
  ROUND(AVG(value), 2) AS avg_value,
  ROUND(MIN(value), 2) AS min_value,
  ROUND(MAX(value), 2) AS max_value
FROM `my-new-project-473015.my_eicu_derived.sepsis3_gcs_24h_raw`
GROUP BY variable_name
ORDER BY variable_name

,variable_name,count,patients,avg_value,min_value,max_value
0,gcs_eye,148059,22147,3.23,1.0,4.0
1,gcs_motor,147867,22142,5.32,1.0,6.0
2,gcs_verbal,145243,21985,3.25,1.0,5.0


## STEP 12: Combine All Tables

Goal: Combine all extracted data into one unified long-format table

Tables to combine:
1. sepsis3_vitals_24h_raw (hr, map, temp, rr)
2. sepsis3_fio2_24h_raw (fio2)
3. sepsis3_labs_24h_raw (sodium, bun, hct, wbc, albumin, bili, glucose, scr)
4. sepsis3_bloodgas_24h_raw (ph, pao2, pco2, aa_grad)
5. sepsis3_gcs_24h_raw (gcs_eye, gcs_motor, gcs_verbal)

In [85]:
%%bigquery
CREATE OR REPLACE TABLE `my-new-project-473015.my_eicu_derived.sepsis3_apache_vars_24h_raw` AS

-- Vital Signs
SELECT
  patientunitstayid,
  observationoffset,
  variable_name,
  value,
  'vitalsign' AS data_source
FROM `my-new-project-473015.my_eicu_derived.sepsis3_vitals_24h_raw`

UNION ALL

-- FiO2
SELECT
  patientunitstayid,
  observationoffset,
  variable_name,
  value,
  CONCAT('fio2_', source) AS data_source
FROM `my-new-project-473015.my_eicu_derived.sepsis3_fio2_24h_raw`

UNION ALL

-- Lab Values
SELECT
  patientunitstayid,
  observationoffset,
  variable_name,
  value,
  'labs' AS data_source
FROM `my-new-project-473015.my_eicu_derived.sepsis3_labs_24h_raw`

UNION ALL

-- Blood Gas
SELECT
  patientunitstayid,
  observationoffset,
  variable_name,
  value,
  'bloodgas' AS data_source
FROM `my-new-project-473015.my_eicu_derived.sepsis3_bloodgas_24h_raw`

UNION ALL

-- GCS Components
SELECT
  patientunitstayid,
  observationoffset,
  variable_name,
  value,
  'gcs' AS data_source
FROM `my-new-project-473015.my_eicu_derived.sepsis3_gcs_24h_raw`

ORDER BY patientunitstayid, observationoffset, variable_name;

SELECT 'Table sepsis3_apache_vars_24h_raw created successfully!' AS status;

,status
0,Table sepsis3_apache_vars_24h_raw created succ...


In [86]:
%%bigquery
-- Verify combined data
SELECT
  data_source,
  variable_name,
  COUNT(*) AS count,
  COUNT(DISTINCT patientunitstayid) AS patients
FROM `my-new-project-473015.my_eicu_derived.sepsis3_apache_vars_24h_raw`
GROUP BY data_source, variable_name
ORDER BY data_source, variable_name

,data_source,variable_name,count,patients
0,bloodgas,aa_grad,24482,12977
1,bloodgas,pao2,61140,20846
2,bloodgas,pco2,42941,13252
3,bloodgas,ph,73242,20150
4,fio2_apacheapsvar,fio2,12770,12770
5,fio2_respiratorycharting,fio2,251819,18962
6,gcs,gcs_eye,148059,22147
7,gcs,gcs_motor,147867,22142
8,gcs,gcs_verbal,145243,21985
9,labs,albumin,37454,22531


## STEP 13: Add Age and Comorbidity

Goal: Add age and comorbidity information to the cohort

**eICU Notes:**
- Age in eICU is stored as STRING with "> 89" for elderly patients
- Comorbidities from ICD codes in `diagnosis` table

In [87]:
%%bigquery
CREATE OR REPLACE TABLE `my-new-project-473015.my_eicu_derived.sepsis3_cohort_final` AS
WITH patient_info AS (
  SELECT
    c.patientunitstayid,
    p.gender AS sex
  FROM `my-new-project-473015.my_eicu_derived.sepsis3_cohort_with_vent_uop_admit_arf` c
  INNER JOIN `physionet-data.eicu_crd.patient` p
    ON c.patientunitstayid = p.patientunitstayid
),
-- Priority 1: Get comorbidity from apachepredvar (most reliable)
apache_comorbidity AS (
  SELECT
    patientunitstayid,
    -- Priority order: aids > hepaticfailure > lymphoma > metastaticcancer > leukemia > immunosuppression > cirrhosis
    CASE
      WHEN aids = 1 THEN 'aids'
      WHEN hepaticfailure = 1 THEN 'hepatic_failure'
      WHEN lymphoma = 1 THEN 'lymphoma'
      WHEN metastaticcancer = 1 THEN 'cancer_mets'
      WHEN leukemia = 1 THEN 'leukemia'
      WHEN immunosuppression = 1 THEN 'immunosuppress'
      WHEN cirrhosis = 1 THEN 'cirrhosis'
      ELSE NULL
    END AS comorbidity_apache
  FROM `physionet-data.eicu_crd.apachepredvar`
),
-- Priority 2: Fallback to ICD codes from diagnosis table
icd_comorbidity AS (
  SELECT
    c.patientunitstayid,
    CASE
      -- AIDS
      WHEN MAX(CASE
        WHEN LOWER(d.icd9code) LIKE '042%' OR LOWER(d.icd9code) LIKE '043%' OR LOWER(d.icd9code) LIKE '044%' THEN 1
        WHEN LOWER(d.icd9code) LIKE 'b20%' OR LOWER(d.icd9code) LIKE 'b21%' OR LOWER(d.icd9code) LIKE 'b22%' THEN 1
        ELSE 0 END) = 1
        THEN 'aids'
      -- Hepatic failure
      WHEN MAX(CASE
        WHEN LOWER(d.icd9code) LIKE '5722%' OR LOWER(d.icd9code) LIKE '5723%' OR LOWER(d.icd9code) LIKE '5724%' THEN 1
        WHEN LOWER(d.icd9code) LIKE '4560%' OR LOWER(d.icd9code) LIKE '4561%' OR LOWER(d.icd9code) LIKE '4562%' THEN 1
        WHEN LOWER(d.diagnosisstring) LIKE '%hepatic failure%' THEN 1
        WHEN LOWER(d.diagnosisstring) LIKE '%liver failure%' THEN 1
        ELSE 0 END) = 1
        THEN 'hepatic_failure'
      -- Lymphoma
      WHEN MAX(CASE
        WHEN LOWER(d.icd9code) LIKE '200%' OR LOWER(d.icd9code) LIKE '201%' OR LOWER(d.icd9code) LIKE '202%' THEN 1
        WHEN LOWER(d.diagnosisstring) LIKE '%lymphoma%' THEN 1
        ELSE 0 END) = 1
        THEN 'lymphoma'
      -- Metastatic cancer
      WHEN MAX(CASE
        WHEN LOWER(d.icd9code) LIKE '196%' OR LOWER(d.icd9code) LIKE '197%' OR LOWER(d.icd9code) LIKE '198%' OR LOWER(d.icd9code) LIKE '199%' THEN 1
        WHEN LOWER(d.diagnosisstring) LIKE '%metasta%' THEN 1
        ELSE 0 END) = 1
        THEN 'cancer_mets'
      -- Leukemia
      WHEN MAX(CASE
        WHEN LOWER(d.icd9code) LIKE '204%' OR LOWER(d.icd9code) LIKE '205%' OR LOWER(d.icd9code) LIKE '206%' OR LOWER(d.icd9code) LIKE '207%' OR LOWER(d.icd9code) LIKE '208%' THEN 1
        WHEN LOWER(d.diagnosisstring) LIKE '%leukemia%' THEN 1
        ELSE 0 END) = 1
        THEN 'leukemia'
      -- Immunosuppression
      WHEN MAX(CASE
        WHEN LOWER(d.diagnosisstring) LIKE '%immunosuppress%' THEN 1
        WHEN LOWER(d.diagnosisstring) LIKE '%immunodeficien%' THEN 1
        ELSE 0 END) = 1
        THEN 'immunosuppress'
      -- Cirrhosis
      WHEN MAX(CASE
        WHEN LOWER(d.icd9code) LIKE '5712%' OR LOWER(d.icd9code) LIKE '5715%' OR LOWER(d.icd9code) LIKE '5716%' THEN 1
        WHEN LOWER(d.diagnosisstring) LIKE '%cirrhosis%' THEN 1
        ELSE 0 END) = 1
        THEN 'cirrhosis'
      ELSE NULL
    END AS comorbidity_icd
  FROM `my-new-project-473015.my_eicu_derived.sepsis3_cohort_with_vent_uop_admit_arf` c
  LEFT JOIN `physionet-data.eicu_crd.diagnosis` d
    ON c.patientunitstayid = d.patientunitstayid
  GROUP BY c.patientunitstayid
),
mortality AS (
  SELECT
    c.patientunitstayid,
    CASE
      WHEN p.hospitaldischargestatus = 'Expired' THEN 1
      ELSE 0
    END AS hospital_expire_flag
  FROM `my-new-project-473015.my_eicu_derived.sepsis3_cohort_with_vent_uop_admit_arf` c
  INNER JOIN `physionet-data.eicu_crd.patient` p
    ON c.patientunitstayid = p.patientunitstayid
)
SELECT
  s.*,
  pi.sex,
  -- Use apachepredvar first, fallback to ICD codes
  COALESCE(ac.comorbidity_apache, ic.comorbidity_icd) AS comorbidity,
  m.hospital_expire_flag
FROM `my-new-project-473015.my_eicu_derived.sepsis3_cohort_with_vent_uop_admit_arf` s
LEFT JOIN patient_info pi
  ON s.patientunitstayid = pi.patientunitstayid
LEFT JOIN apache_comorbidity ac
  ON s.patientunitstayid = ac.patientunitstayid
LEFT JOIN icd_comorbidity ic
  ON s.patientunitstayid = ic.patientunitstayid
LEFT JOIN mortality m
  ON s.patientunitstayid = m.patientunitstayid;

SELECT 'Table sepsis3_cohort_final created successfully!' AS status;

,status
0,Table sepsis3_cohort_final created successfully!


In [88]:
%%bigquery
SELECT column_name
FROM `my-new-project-473015.my_eicu_derived.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name = 'sepsis3_cohort_with_vent_uop_admit_arf'
ORDER BY ordinal_position

,column_name
0,patientunitstayid
1,uniquepid
2,hospitalid
3,age
4,gender
5,ethnicity
6,unittype
7,unitvisitnumber
8,hospitaladmitoffset
9,hospitaldischargeoffset


In [89]:
%%bigquery
-- Verify final cohort
SELECT
  COUNT(*) AS total_patients,
  ROUND(AVG(age), 1) AS avg_age,
  SUM(CASE WHEN sex = 'Male' THEN 1 ELSE 0 END) AS male_count,
  SUM(CASE WHEN hospital_expire_flag = 1 THEN 1 ELSE 0 END) AS mortality_count,
  ROUND(100.0 * SUM(CASE WHEN hospital_expire_flag = 1 THEN 1 ELSE 0 END) / COUNT(*), 1) AS mortality_pct
FROM `my-new-project-473015.my_eicu_derived.sepsis3_cohort_final`

,total_patients,avg_age,male_count,mortality_count,mortality_pct
0,31410,65.2,17410,4371,13.9


## STEP 14: Aggregate Values (Wide Format)

Goal: Calculate aggregated values (max/min) from long format data
- Create wide-format table with one row per patient
- Extract worst values (max/min) for each APACHE III variable

In [90]:
%%bigquery
CREATE OR REPLACE TABLE `my-new-project-473015.my_eicu_derived.sepsis3_apache3_dataset` AS
WITH aggregated_vars AS (
  SELECT
    patientunitstayid,
    -- Heart Rate
    MAX(CASE WHEN variable_name = 'hr' THEN value END) AS hr_max,
    MIN(CASE WHEN variable_name = 'hr' THEN value END) AS hr_min,
    -- Mean Arterial Pressure
    MAX(CASE WHEN variable_name = 'map' THEN value END) AS map_max,
    MIN(CASE WHEN variable_name = 'map' THEN value END) AS map_min,
    -- Temperature
    MAX(CASE WHEN variable_name = 'temp' THEN value END) AS temp_max,
    MIN(CASE WHEN variable_name = 'temp' THEN value END) AS temp_min,
    -- Respiratory Rate
    MAX(CASE WHEN variable_name = 'rr' THEN value END) AS rr_max,
    MIN(CASE WHEN variable_name = 'rr' THEN value END) AS rr_min,
    -- FiO2 (max only)
    MAX(CASE WHEN variable_name = 'fio2' THEN value END) AS fio2,
    -- Sodium
    MAX(CASE WHEN variable_name = 'sodium' THEN value END) AS sodium_max,
    MIN(CASE WHEN variable_name = 'sodium' THEN value END) AS sodium_min,
    -- BUN (max only)
    MAX(CASE WHEN variable_name = 'bun' THEN value END) AS bun,
    -- Hematocrit
    MAX(CASE WHEN variable_name = 'hct' THEN value END) AS hct_max,
    MIN(CASE WHEN variable_name = 'hct' THEN value END) AS hct_min,
    -- WBC
    MAX(CASE WHEN variable_name = 'wbc' THEN value END) AS wbc_max,
    MIN(CASE WHEN variable_name = 'wbc' THEN value END) AS wbc_min,
    -- Albumin
    MAX(CASE WHEN variable_name = 'albumin' THEN value END) AS albumin_max,
    MIN(CASE WHEN variable_name = 'albumin' THEN value END) AS albumin_min,
    -- Bilirubin (max only)
    MAX(CASE WHEN variable_name = 'bili' THEN value END) AS bili,
    -- Glucose
    MAX(CASE WHEN variable_name = 'glucose' THEN value END) AS glucose_max,
    MIN(CASE WHEN variable_name = 'glucose' THEN value END) AS glucose_min,
    -- Serum Creatinine
    MAX(CASE WHEN variable_name = 'scr' THEN value END) AS scr_max,
    MIN(CASE WHEN variable_name = 'scr' THEN value END) AS scr_min,
    -- pH
    MAX(CASE WHEN variable_name = 'ph' THEN value END) AS ph_max,
    MIN(CASE WHEN variable_name = 'ph' THEN value END) AS ph_min,
    -- PaO2 (min only)
    MIN(CASE WHEN variable_name = 'pao2' THEN value END) AS pao2,
    -- PCO2
    MAX(CASE WHEN variable_name = 'pco2' THEN value END) AS pco2_max,
    MIN(CASE WHEN variable_name = 'pco2' THEN value END) AS pco2_min,
    -- A-a gradient (max only)
    MAX(CASE WHEN variable_name = 'aa_grad' THEN value END) AS aa_grad,
    -- GCS components (min = worst)
    MIN(CASE WHEN variable_name = 'gcs_eye' THEN value END) AS gcs_eye,
    MIN(CASE WHEN variable_name = 'gcs_motor' THEN value END) AS gcs_motor,
    MIN(CASE WHEN variable_name = 'gcs_verbal' THEN value END) AS gcs_verbal
  FROM `my-new-project-473015.my_eicu_derived.sepsis3_apache_vars_24h_raw`
  GROUP BY patientunitstayid
)
SELECT
  c.patientunitstayid,
  c.uniquepid,
  c.age,
  c.sex,
  c.vent,
  c.uop,
  c.admit,
  c.arf,
  c.scr_max AS scr_cohort,
  c.comorbidity,
  c.hospital_expire_flag,
  -- Aggregated variables from raw data
  av.hr_max,
  av.hr_min,
  av.map_max,
  av.map_min,
  av.temp_max,
  av.temp_min,
  av.rr_max,
  av.rr_min,
  av.fio2,
  av.sodium_max,
  av.sodium_min,
  av.bun,
  av.hct_max,
  av.hct_min,
  av.wbc_max,
  av.wbc_min,
  av.albumin_max,
  av.albumin_min,
  av.bili,
  av.glucose_max,
  av.glucose_min,
  av.scr_max,
  av.scr_min,
  av.ph_max,
  av.ph_min,
  av.pao2,
  av.pco2_max,
  av.pco2_min,
  av.aa_grad,
  av.gcs_eye,
  av.gcs_motor,
  av.gcs_verbal
FROM `my-new-project-473015.my_eicu_derived.sepsis3_cohort_final` c
LEFT JOIN aggregated_vars av
  ON c.patientunitstayid = av.patientunitstayid;

SELECT 'Table sepsis3_apache3_dataset created successfully!' AS status;

,status
0,Table sepsis3_apache3_dataset created successf...


In [91]:
%%bigquery
-- Verify aggregated dataset
SELECT
  COUNT(*) AS total_patients,
  COUNT(hr_max) AS has_hr,
  COUNT(map_min) AS has_map,
  COUNT(temp_max) AS has_temp,
  COUNT(gcs_eye) AS has_gcs,
  COUNT(pao2) AS has_pao2
FROM `my-new-project-473015.my_eicu_derived.sepsis3_apache3_dataset`

,total_patients,has_hr,has_map,has_temp,has_gcs,has_pao2
0,31410,30768,30826,30972,22147,20846


## STEP 15: Calculate APACHE III Scores

Apply APACHE III scoring rules based on CHEST 1991 paper and MIMIC SQL implementation.

**Key Scoring Logic Points:**
1. Pulmonary Score: Uses PaO2 for non-ventilated, A-a gradient for ventilated patients
2. Comorbidity points only added for emergency admissions

In [92]:
%%bigquery
CREATE OR REPLACE TABLE `my-new-project-473015.my_eicu_derived.sepsis3_apache3_scores` AS
WITH scored_data AS (
  SELECT
    *,

    -- Heart Rate Score
    CASE
      WHEN hr_max >= 155 THEN 17
      WHEN hr_max >= 140 THEN 13
      WHEN hr_max >= 120 THEN 7
      WHEN hr_max >= 110 THEN 5
      WHEN hr_min <= 39 THEN 8
      WHEN hr_min <= 49 THEN 5
      WHEN hr_max >= 100 THEN 1
      ELSE 0
    END AS hr_score,

    -- Mean Arterial Pressure Score
    CASE
      WHEN map_min < 40 THEN 23
      WHEN map_min < 60 THEN 15
      WHEN map_min < 70 THEN 7
      WHEN map_min < 80 THEN 6
      WHEN map_min < 100 THEN 0
      WHEN map_max < 120 THEN 4
      WHEN map_max < 130 THEN 7
      WHEN map_max < 140 THEN 9
      WHEN map_max >= 140 THEN 10
      ELSE 0
    END AS map_score,

    -- Temperature Score (Celsius)
    CASE
      WHEN temp_min < 33.0 THEN 28
      WHEN temp_min < 33.5 THEN 16
      WHEN temp_min < 34.0 THEN 13
      WHEN temp_min < 35.0 THEN 8
      WHEN temp_min < 36.0 THEN 2
      WHEN temp_max >= 40.0 THEN 4
      ELSE 0
    END AS temp_score,

    -- Respiratory Rate Score
    CASE
      WHEN vent = TRUE AND rr_min >= 6 AND rr_max <= 12 THEN 0
      WHEN rr_max >= 50 THEN 18
      WHEN rr_min < 6 THEN 17
      WHEN rr_max >= 40 THEN 11
      WHEN rr_max >= 35 THEN 9
      WHEN rr_min < 12 THEN 8
      WHEN rr_min < 14 THEN 7
      WHEN rr_max >= 25 THEN 6
      ELSE 0
    END AS rr_score,

    -- Sodium Score
    CASE
      WHEN sodium_max >= 155 THEN 4
      WHEN sodium_min <= 119 THEN 3
      WHEN sodium_min <= 134 THEN 2
      ELSE 0
    END AS sodium_score,

    -- BUN Score
    CASE
      WHEN bun >= 80 THEN 12
      WHEN bun >= 40 THEN 11
      WHEN bun >= 20 THEN 7
      WHEN bun >= 17 THEN 2
      ELSE 0
    END AS bun_score,

    -- Hematocrit Score
    CASE
      WHEN hct_min < 41 THEN 3
      WHEN hct_max >= 50 THEN 3
      ELSE 0
    END AS hct_score,

    -- WBC Score
    CASE
      WHEN wbc_min < 1 THEN 19
      WHEN wbc_max >= 25 OR wbc_min <= 2.9 THEN 5
      WHEN wbc_max >= 20 THEN 1
      ELSE 0
    END AS wbc_score,

    -- Bilirubin Score
    CASE
      WHEN bili >= 8 THEN 16
      WHEN bili >= 5 THEN 8
      WHEN bili >= 3 THEN 6
      WHEN bili >= 2 THEN 5
      ELSE 0
    END AS bili_score,

    -- Albumin Score
    CASE
      WHEN albumin_min <= 1.9 THEN 11
      WHEN albumin_min <= 2.4 THEN 6
      WHEN albumin_max >= 4.5 THEN 4
      ELSE 0
    END AS albumin_score,

    -- Glucose Score
    CASE
      WHEN glucose_min <= 39 THEN 8
      WHEN glucose_min <= 59 THEN 9
      WHEN glucose_max >= 350 THEN 5
      WHEN glucose_max >= 200 THEN 3
      ELSE 0
    END AS glucose_score,

    -- Creatinine Score (with ARF consideration)
    CASE
      WHEN arf = TRUE AND scr_max >= 1.5 THEN 10
      WHEN scr_max >= 1.95 THEN 7
      WHEN scr_max >= 1.5 THEN 4
      WHEN scr_min < 0.4 THEN 3
      ELSE 0
    END AS scr_score,

    -- pH Score (with PCO2 consideration)
    CASE
      WHEN ph_min < 7.2 THEN 12
      WHEN ph_min < 7.3 AND pco2_min < 25 THEN 12
      WHEN ph_min < 7.3 THEN 6
      WHEN ph_min < 7.35 AND pco2_min < 25 THEN 6
      WHEN ph_min < 7.35 AND pco2_min < 40 THEN 2
      WHEN ph_max >= 7.55 AND pco2_max >= 45 THEN 12
      WHEN ph_max >= 7.55 AND pco2_max < 45 THEN 3
      WHEN ph_max >= 7.5 AND pco2_max >= 45 THEN 5
      WHEN ph_max >= 7.5 AND pco2_max < 45 THEN 1
      ELSE 0
    END AS ph_score,

    -- Pulmonary Score (PaO2 vs A-a gradient based on ventilation and FiO2)
    CASE
      WHEN vent = TRUE AND fio2 >= 50 THEN
        CASE
          WHEN aa_grad >= 500 THEN 14
          WHEN aa_grad >= 350 THEN 11
          WHEN aa_grad >= 250 THEN 9
          WHEN aa_grad >= 100 THEN 4
          ELSE 0
        END
      ELSE
        CASE
          WHEN pao2 < 50 THEN 15
          WHEN pao2 < 55 THEN 8
          WHEN pao2 < 60 THEN 5
          WHEN pao2 < 70 THEN 3
          WHEN pao2 < 80 THEN 1
          ELSE 0
        END
    END AS pulmonary_score,

    -- GCS Score
    CASE
      WHEN COALESCE(gcs_eye, 4) + COALESCE(gcs_motor, 6) + COALESCE(gcs_verbal, 5) <= 5 THEN 48
      WHEN COALESCE(gcs_eye, 4) + COALESCE(gcs_motor, 6) + COALESCE(gcs_verbal, 5) <= 7 THEN 33
      WHEN COALESCE(gcs_eye, 4) + COALESCE(gcs_motor, 6) + COALESCE(gcs_verbal, 5) <= 9 THEN 23
      WHEN COALESCE(gcs_eye, 4) + COALESCE(gcs_motor, 6) + COALESCE(gcs_verbal, 5) <= 11 THEN 16
      WHEN COALESCE(gcs_eye, 4) + COALESCE(gcs_motor, 6) + COALESCE(gcs_verbal, 5) <= 13 THEN 5
      ELSE 0
    END AS gcs_score,

    -- Age Score
    CASE
      WHEN age >= 85 THEN 24
      WHEN age >= 75 THEN 17
      WHEN age >= 65 THEN 13
      WHEN age >= 55 THEN 11
      WHEN age >= 45 THEN 5
      ELSE 0
    END AS age_score,

    -- Comorbidity Score (only for emergency admissions)
    CASE
      WHEN admit = 'emergency' THEN
        CASE
          WHEN comorbidity = 'aids' THEN 23
          WHEN comorbidity = 'hepatic_failure' THEN 16
          WHEN comorbidity = 'lymphoma' THEN 13
          WHEN comorbidity = 'cancer_mets' THEN 11
          WHEN comorbidity = 'leukemia' THEN 10
          WHEN comorbidity = 'mult_myeloma' THEN 10
          WHEN comorbidity = 'immunosuppress' THEN 10
          WHEN comorbidity = 'cirrhosis' THEN 4
          ELSE 0
        END
      ELSE 0
    END AS comorbidity_score

  FROM `my-new-project-473015.my_eicu_derived.sepsis3_apache3_dataset`
)
SELECT
  *,
  -- Total APACHE III Score
  COALESCE(hr_score, 0) +
  COALESCE(map_score, 0) +
  COALESCE(temp_score, 0) +
  COALESCE(rr_score, 0) +
  COALESCE(sodium_score, 0) +
  COALESCE(bun_score, 0) +
  COALESCE(hct_score, 0) +
  COALESCE(wbc_score, 0) +
  COALESCE(bili_score, 0) +
  COALESCE(albumin_score, 0) +
  COALESCE(glucose_score, 0) +
  COALESCE(scr_score, 0) +
  COALESCE(ph_score, 0) +
  COALESCE(pulmonary_score, 0) +
  COALESCE(gcs_score, 0) +
  COALESCE(age_score, 0) +
  COALESCE(comorbidity_score, 0) AS apache3_score
FROM scored_data;

SELECT 'Table sepsis3_apache3_scores created successfully!' AS status;

,status
0,Table sepsis3_apache3_scores created successfu...


In [93]:
%%bigquery
-- Verify APACHE III scores
SELECT
  COUNT(*) AS total_patients,
  ROUND(AVG(apache3_score), 1) AS avg_score,
  ROUND(MIN(apache3_score), 1) AS min_score,
  ROUND(MAX(apache3_score), 1) AS max_score,
  ROUND(STDDEV(apache3_score), 1) AS std_score
FROM `my-new-project-473015.my_eicu_derived.sepsis3_apache3_scores`

,total_patients,avg_score,min_score,max_score,std_score
0,31410,69.4,4.0,208.0,28.8


## STEP 16: Combine with Vital Signs Data

Goal: Create final table combining APACHE III scores with time-series vital signs data

In [94]:
%%bigquery
CREATE OR REPLACE TABLE `my-new-project-473015.my_eicu_derived.sepsis3_apache3_with_vitals` AS
SELECT
  -- Patient identifiers
  s.uniquepid,
  s.patientunitstayid,

  -- Demographics
  s.age,
  s.sex,

  -- Clinical status
  s.vent,
  s.uop,
  s.admit,
  s.arf,
  s.comorbidity,

  -- Severity score
  s.apache3_score,

  -- Outcome
  s.hospital_expire_flag,

  -- Time-series measurements
  v.observationoffset,
  v.variable_name,
  v.value,
  v.data_source

FROM `my-new-project-473015.my_eicu_derived.sepsis3_apache3_scores` s
LEFT JOIN `my-new-project-473015.my_eicu_derived.sepsis3_apache_vars_24h_raw` v
  ON s.patientunitstayid = v.patientunitstayid;

SELECT 'Table sepsis3_apache3_with_vitals created successfully!' AS status;

,status
0,Table sepsis3_apache3_with_vitals created succ...


In [95]:
%%bigquery
-- Update sex to M/F format to match MIMIC-IV, set Unknown/Other/blank to NULL
CREATE OR REPLACE TABLE `my-new-project-473015.my_eicu_derived.sepsis3_apache3_with_vitals` AS
SELECT
  uniquepid,
  patientunitstayid,
  age,
  CASE
    WHEN sex = 'Male' THEN 'M'
    WHEN sex = 'Female' THEN 'F'
    WHEN sex = 'Unknown' THEN NULL
    WHEN sex = 'Other' THEN NULL
    WHEN sex = '' THEN NULL
    WHEN sex IS NULL THEN NULL
    ELSE NULL
  END AS sex,
  vent,
  uop,
  admit,
  arf,
  comorbidity,
  apache3_score,
  hospital_expire_flag,
  observationoffset,
  variable_name,
  value,
  data_source
FROM `my-new-project-473015.my_eicu_derived.sepsis3_apache3_with_vitals`;

SELECT 'Sex updated to M/F format!' AS status;

,status
0,Sex updated to M/F format!


In [96]:
%%bigquery
-- Verify sex values
SELECT sex, COUNT(DISTINCT patientunitstayid) as patient_count
FROM `my-new-project-473015.my_eicu_derived.sepsis3_apache3_with_vitals`
GROUP BY sex

,sex,patient_count
0,F,13997
1,M,17410
2,None,3


In [97]:
%%bigquery
-- Verify final combined table
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT patientunitstayid) AS unique_patients,
  COUNT(DISTINCT variable_name) AS unique_variables
FROM `my-new-project-473015.my_eicu_derived.sepsis3_apache3_with_vitals`

,total_rows,unique_patients,unique_variables
0,5473554,31410,20


---

## 📊 Summary: Tables Created

| Table Name | Description | Corresponding MIMIC-IV Table |
|------------|-------------|-----------------------------|
| `sepsis3_cohort_with_vent` | Cohort + ventilation data | `sepsis3_cohort_with_vent` |
| `sepsis3_cohort_with_vent_uop` | + urine output | `sepsis3_cohort_with_vent_uop` |
| `sepsis3_cohort_with_vent_uop_admit` | + admission type | `sepsis3_cohort_with_vent_uop_admit` |
| `sepsis3_cohort_with_vent_uop_admit_arf` | + ARF flag | `sepsis3_cohort_with_vent_uop_admit_arf` |
| `sepsis3_vitals_24h_raw` | Vital signs (long format) | `sepsis3_vitals_24h_raw` |
| `sepsis3_fio2_24h_raw` | FiO2 (long format) | `sepsis3_fio2_24h_raw` |
| `sepsis3_labs_24h_raw` | Lab values (long format) | `sepsis3_labs_24h_raw` |
| `sepsis3_bloodgas_24h_raw` | Blood gas (long format) | `sepsis3_bloodgas_24h_raw` |
| `sepsis3_gcs_24h_raw` | GCS components (long format) | `sepsis3_gcs_24h_raw` |
| `sepsis3_apache_vars_24h_raw` | Combined raw data | `sepsis3_apache_vars_24h_raw` |
| `sepsis3_cohort_final` | Final cohort with demographics | `sepsis3_cohort_final` |
| `sepsis3_apache3_dataset` | Aggregated wide format | `sepsis3_apache3_dataset` |
| `sepsis3_apache3_scores` | APACHE III scores | `sepsis3_apache3_scores` |
| `sepsis3_apache3_with_vitals` | Final combined table | `sepsis3_apache3_with_vitals` |

---

## 🔑 Key Differences from MIMIC-IV Implementation

1. **Time Reference:**
   - MIMIC-IV uses DATETIME
   - eICU uses offset in minutes (0 = ICU admission, 1440 = 24h)

2. **Patient Identifiers:**
   - MIMIC-IV: `subject_id`, `hadm_id`, `stay_id`
   - eICU: `uniquepid`, `patientunitstayid`

3. **Age:**
   - MIMIC-IV: Numeric (anchor_age)
   - eICU: String (includes "> 89" for elderly)

4. **Mortality:**
   - MIMIC-IV: `hospital_expire_flag` from admissions
   - eICU: `hospitaldischargestatus = 'Expired'`

5. **Data Sources:**
   - MIMIC-IV: `mimiciv_derived` tables
   - eICU: `eicu_crd_derived` + custom `my_eicu_derived` tables

In [98]:
%%bigquery
-- Check all column names in final table
SELECT column_name, data_type
FROM `my-new-project-473015.my_eicu_derived.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name = 'sepsis3_apache3_with_vitals'
ORDER BY ordinal_position

,column_name,data_type
0,uniquepid,STRING
1,patientunitstayid,INT64
2,age,INT64
3,sex,STRING
4,vent,BOOL
5,uop,FLOAT64
6,admit,STRING
7,arf,BOOL
8,comorbidity,STRING
9,apache3_score,INT64


In [101]:
%%bigquery
-- Check variable_name distribution with statistics
SELECT
  variable_name,
  COUNT(*) as record_count,
  COUNT(DISTINCT patientunitstayid) as patient_count,
  ROUND(COUNT(*) / COUNT(DISTINCT patientunitstayid), 1) as avg_measurements_per_patient,
  ROUND(MIN(value), 2) as min_value,
  ROUND(AVG(value), 2) as avg_value,
  ROUND(APPROX_QUANTILES(value, 100)[OFFSET(50)], 2) as median_value,
  ROUND(MAX(value), 2) as max_value,
  ROUND(STDDEV(value), 2) as std_value
FROM `my-new-project-473015.my_eicu_derived.sepsis3_apache3_with_vitals`
WHERE variable_name IS NOT NULL
GROUP BY variable_name
ORDER BY variable_name

,variable_name,record_count,patient_count,avg_measurements_per_patient,min_value,avg_value,median_value,max_value,std_value
0,aa_grad,24482,12977,1.9,0.00,252.81,207.40,650.38,163.12
1,albumin,37454,22531,1.7,0.20,2.88,2.90,6.20,0.71
2,bili,35239,21870,1.6,0.00,1.43,0.70,48.00,2.94
3,bun,77764,30709,2.5,0.00,31.59,24.00,298.00,24.60
4,fio2,264589,20922,12.6,21.00,53.19,50.00,100.00,21.35
5,gcs_eye,148059,22147,6.7,1.00,3.23,4.00,4.00,1.07
6,gcs_motor,147867,22142,6.7,1.00,5.32,6.00,6.00,1.35
7,gcs_verbal,145243,21985,6.6,1.00,3.25,4.00,5.00,1.83
8,glucose,90345,30684,2.9,2.00,156.31,135.00,998.00,86.12
9,hct,93501,30956,3.0,10.00,32.05,31.50,72.40,7.22


In [102]:
%%bigquery
-- Check potential outliers
SELECT
  variable_name,
  COUNTIF(value <= 0) as zero_or_negative,
  -- MAP outliers
  COUNTIF(variable_name = 'map' AND value < 20) as map_below_20,
  -- Temp outliers
  COUNTIF(variable_name = 'temp' AND value < 30) as temp_below_30,
  -- WBC outliers
  COUNTIF(variable_name = 'wbc' AND value > 100) as wbc_above_100,
  -- PaO2 outliers
  COUNTIF(variable_name = 'pao2' AND value < 30) as pao2_below_30
FROM `my-new-project-473015.my_eicu_derived.sepsis3_apache3_with_vitals`
WHERE variable_name IN ('map', 'temp', 'wbc', 'pao2', 'bili')
GROUP BY variable_name

,variable_name,zero_or_negative,map_below_20,temp_below_30,wbc_above_100,pao2_below_30
0,map,0,339,0,0,0
1,wbc,17,0,0,56,0
2,bili,1,0,0,0,0
3,temp,0,0,86,0,0
4,pao2,0,0,0,0,181


In [103]:
%%bigquery
-- Check extreme value distributions
SELECT
  variable_name,
  ROUND(APPROX_QUANTILES(value, 100)[OFFSET(1)], 2) as p1,
  ROUND(APPROX_QUANTILES(value, 100)[OFFSET(5)], 2) as p5,
  ROUND(APPROX_QUANTILES(value, 100)[OFFSET(95)], 2) as p95,
  ROUND(APPROX_QUANTILES(value, 100)[OFFSET(99)], 2) as p99
FROM `my-new-project-473015.my_eicu_derived.sepsis3_apache3_with_vitals`
WHERE variable_name IN ('map', 'temp', 'wbc', 'pao2', 'bili')
GROUP BY variable_name
ORDER BY variable_name

,variable_name,p1,p5,p95,p99
0,bili,0.2,0.20,4.8,15.4
1,map,46.0,55.00,107.0,125.0
2,pao2,39.0,58.00,418.0,514.0
3,temp,32.9,34.90,38.4,39.2
4,wbc,1.0,4.06,27.7,41.5


In [108]:
%%bigquery
-- Check what data these patients have in sepsis3_apache3_dataset
SELECT
  patientunitstayid,
  age,
  hr_max, temp_max, map_max,
  sodium_max, bun, bili, aa_grad
FROM `my-new-project-473015.my_eicu_derived.sepsis3_apache3_dataset`
WHERE patientunitstayid IN (312197, 562366, 597728, 196992, 196023, 696367, 680718)

,patientunitstayid,age,hr_max,temp_max,map_max,sodium_max,bun,bili,aa_grad


In [107]:
%%bigquery
-- Check if these patients exist in sepsis3_apache_vars_24h_raw
SELECT patientunitstayid, COUNT(*) as cnt
FROM `my-new-project-473015.my_eicu_derived.sepsis3_apache_vars_24h_raw`
WHERE patientunitstayid IN (312197, 562366, 597728, 196992, 196023, 696367, 680718)
GROUP BY patientunitstayid

,patientunitstayid,cnt
